# 30 · Stopping conditions: step limits, timeouts and cancel
Level L3 · Part 4 · core · ~30 min · builds on 26

## Goal
Stop Scout's agent loop cleanly in four ways: a step limit, a wall-clock timeout, repeated identical
tool calls, and a user cancel. Every stop returns the history so far and says *why* it stopped.

## The idea
Lesson 26's loop has one guard, the step limit. That is not enough. Ten steps can take ten minutes if
each call is slow. A model can ask for the same search again and again, wasting steps until the limit.
And a person watching may want to stop the run now.

So the loop asks four questions. Before each model call: cancelled? out of time? After the tools run:
same call again? out of steps? Any "yes" ends the run with a named reason, never with a crash. Like a
taxi ride, which ends when you say "let me out", the fare cap is hit, the driver circles the same
block, or the meter's distance runs out.

## Picture
```mermaid
flowchart TD
    S[start of a step] --> X{cancelled?}
    X -- yes --> SC[stop: cancelled]
    X -- no --> T{out of time?}
    T -- yes --> ST[stop: timeout]
    T -- no --> M[call the model]
    M --> A{tool calls?}
    A -- no --> E[stop: end_of_turn]
    A -- yes --> R[run every tool, append results]
    R --> P{same call too often?}
    P -- yes --> SR[stop: repeated_call]
    P -- no --> L{steps left?}
    L -- yes --> S
    L -- no --> SL[stop: step_limit]
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 26. Without a key the Gemini calls replay from
`../data/llm_cassettes/30_stopping-conditions.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("30_stopping-conditions")

LLM mode: replay (30_stopping-conditions.yaml)


Setup from lesson 26, unchanged: the notes, the client, the pinned model and the two tools.

In [2]:
import ast, json, operator, os, threading, time
from collections import Counter
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

The calculator and both schemas from lesson 26, unchanged.

In [3]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

SCHEMAS = [
    {"name": "search_notes", "description": "Search the user's notes. Returns every note whose text contains "
     "the query word (case-insensitive), as a list of {id, text}. Use it before answering any question "
     "about what the notes say.", "parameters": {"type": "object", "required": ["query"], "properties":
     {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}}}},
    {"name": "calculator", "description": "Evaluate an arithmetic expression with numbers, + - * / and "
     "brackets, e.g. '24*60+39'. Use it for every calculation instead of doing arithmetic yourself.",
     "parameters": {"type": "object", "required": ["expression"],
                    "properties": {"expression": {"type": "string", "description": "e.g. '(3+4)*2'"}}}},
]
TOOLS = {"search_notes": search_notes, "calculator": calculator}
print(calculator("24*60+39"), [s["name"] for s in SCHEMAS])

1479 ['search_notes', 'calculator']


`ScriptedModel` from lesson 26, unchanged.

In [4]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

`run_tool` is lesson 28's version, unchanged: it never raises; a failing tool becomes an error result.

In [5]:
def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "2^3"}}, TOOLS))

{'content': 'ValueError: not allowed in a calculation: BinOp(left=Constant(value=2), op=BitXor(', 'is_error': True}


### 2. Two fake models that never finish
Real models get stuck in two ways. A **stuck** model asks for exactly the same call each time. A
**wandering** model changes its query a little each time ("comet 1", "comet 2", ...), so the calls are
never identical. Both never answer. The script's last entry repeats forever, as in lesson 26.

In [6]:
QUESTION = "How many minutes long is a day on Mars?"
STUCK = [{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}]

def wander(messages: list[dict]) -> dict:
    n = sum(m["role"] == "tool" for m in messages) + 1        # 1, 2, 3, ... one per step
    return {"tool_calls": [{"name": "search_notes", "args": {"query": f"comet {n}"}}]}
WANDERING = [wander]

model = ScriptedModel(WANDERING)
history = [{"role": "user", "text": QUESTION}]
for _ in range(3):
    reply = model(history, SCHEMAS)
    history += [reply, {"role": "tool", "call_id": reply["tool_calls"][0]["id"], "name": "search_notes",
                        "content": "[]", "is_error": False}]
    print(reply["tool_calls"][0]["args"])

{'query': 'comet 1'}
{'query': 'comet 2'}
{'query': 'comet 3'}


### 3. A wall-clock deadline
A **wall-clock timeout** limits the real time a run may take, however many steps that is. We measure
with `time.monotonic()`, which only moves forward. `time.time()` can jump when the computer's clock is
corrected. The clock is a parameter, so a test can pass a fake one and skip the waiting.

In [7]:
class Deadline:
    def __init__(self, seconds: float, clock=time.monotonic):
        self.seconds, self.clock, self.start = seconds, clock, clock()

    def elapsed(self) -> float:
        return round(self.clock() - self.start, 2)

    def expired(self) -> bool:
        return self.elapsed() >= self.seconds

class FakeClock:
    """A clock you move by hand: tests of timeouts without waiting."""
    def __init__(self): self.now = 0.0
    def __call__(self) -> float: return self.now

A slow model is any model whose calls take time. This wrapper makes each call "take" 7 seconds on the
fake clock. With a 20-second deadline, the fourth call should never start.

In [8]:
class SlowModel:
    def __init__(self, model, clock: FakeClock, seconds: float):
        self.model, self.clock, self.seconds = model, clock, seconds

    def __call__(self, messages, tools):
        self.clock.now += self.seconds
        return self.model(messages, tools)

clock = FakeClock()
deadline, slow = Deadline(20, clock), SlowModel(ScriptedModel(WANDERING), clock, seconds=7)
while not deadline.expired():
    slow([{"role": "user", "text": QUESTION}], SCHEMAS)
    print(f"call done at t={deadline.elapsed()} s, expired: {deadline.expired()}")

call done at t=7.0 s, expired: False
call done at t=14.0 s, expired: False
call done at t=21.0 s, expired: True


The deadline is checked *between* calls, so the run can overshoot it by one call (21 s here, not 20).
A single call that hangs is not stopped by this check at all. Step 9 gives each call its own timeout.

### 4. Spotting repeated calls
**Loop detection** catches the stuck model: the same tool with the same arguments, again and again.
A dict cannot go in a `Counter`, so each call becomes a string key. `sort_keys=True` makes
`{"a": 1, "b": 2}` and `{"b": 2, "a": 1}` the same key.

In [9]:
def call_key(call: dict) -> str:
    return call["name"] + json.dumps(call["args"], sort_keys=True)

class RepeatWatch:
    def __init__(self, max_repeats: int):
        self.max_repeats, self.counts = max_repeats, Counter()

    def seen(self, calls: list[dict]) -> bool:
        """Count this step's calls; True once any identical call has run more than max_repeats times."""
        self.counts.update(call_key(c) for c in calls)
        return any(self.counts[call_key(c)] > self.max_repeats for c in calls)

watch = RepeatWatch(max_repeats=2)
print([watch.seen(STUCK[0]["tool_calls"]) for _ in range(4)])
print(call_key({"name": "f", "args": {"a": 1, "b": 2}}) == call_key({"name": "f", "args": {"b": 2, "a": 1}}))

[False, False, True, True]
True


Allowing two is deliberate: re-running a search after writing a file is sometimes right. The wandering
model never trips it, which is why we still need the other guards.

### 5. Cancel: a flag another thread can set
**Cancellation** means the user (a Stop button, a closed browser tab) asks the run to end. The click
usually arrives on another thread, so we use `threading.Event`: one side calls `.set()`, the loop
checks `.is_set()`.

In [10]:
cancel = threading.Event()
print("before:", cancel.is_set())
threading.Thread(target=cancel.set).start()                   # the "Stop" button, on another thread
time.sleep(0.05)
print("after: ", cancel.is_set())

before: False
after:  True


### 6. The guarded loop
Lesson 28's loop, plus the guards. Cancel and the deadline are checked before each model call, the
repeat check after the tools run. Each step finishes running all its tools before any stop, so every
tool call in the history has its result. `finish` builds the same result dict for every way out.
`log_step` prints one line per step so we can watch the runs.

In [11]:
def log_step(step: int, calls: list[dict], verbose: bool) -> None:
    if verbose:
        print(f"  step {step}: " + ", ".join(f"{c['name']}({c['args']})" for c in calls))

The loop itself:

In [12]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6,
              timeout_s: float = 60, max_repeats: int = 2, cancel=None, clock=time.monotonic, verbose=True):
    messages, errors = [{"role": "user", "text": question}], 0
    deadline, repeats = Deadline(timeout_s, clock), RepeatWatch(max_repeats)
    def finish(stop, steps, answer=None):
        return dict(answer=answer, stop=stop, steps=steps, errors=errors, elapsed=deadline.elapsed(),
                    messages=messages)
    for step in range(1, max_steps + 1):
        if cancel is not None and cancel.is_set():           # 1. the user pressed Stop
            return finish("cancelled", step - 1)
        if deadline.expired():                                # 2. out of time
            return finish("timeout", step - 1)
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            return finish("end_of_turn", step, reply["text"])
        for call in calls:                                    # all of them, even if we stop next
            result = run_tool(call, tools)
            errors += result["is_error"]
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
        log_step(step, calls, verbose)
        if repeats.seen(calls):                               # 3. the same call again and again
            return finish("repeated_call", step)
    return finish("step_limit", max_steps)                    # 4. out of steps

### 7. Each guard stops a model that never finishes
Four runs, one per guard. The step limit and the timeout catch the wandering model; loop detection
catches the stuck one far earlier than the step limit would. The cancel run uses a wrapper that
presses Stop during the model's second call, like a user losing patience.

In [13]:
def press_stop_on_call(model, n: int, event: threading.Event):
    def wrapped(messages, tools):
        reply = model(messages, tools)
        if model.calls == n:
            event.set()
        return reply
    return wrapped

clock, stop_button = FakeClock(), threading.Event()
runs = {
    "step limit": run_agent(QUESTION, ScriptedModel(WANDERING), TOOLS, SCHEMAS, max_steps=4),
    "timeout": run_agent(QUESTION, SlowModel(ScriptedModel(WANDERING), clock, 7), TOOLS, SCHEMAS,
                         max_steps=20, timeout_s=20, clock=clock),
    "repeats": run_agent(QUESTION, ScriptedModel(STUCK), TOOLS, SCHEMAS, max_steps=20),
    "cancel": run_agent(QUESTION, press_stop_on_call(ScriptedModel(WANDERING), 2, stop_button),
                        TOOLS, SCHEMAS, max_steps=20, cancel=stop_button),
}

  step 1: search_notes({'query': 'comet 1'})
  step 2: search_notes({'query': 'comet 2'})
  step 3: search_notes({'query': 'comet 3'})
  step 4: search_notes({'query': 'comet 4'})
  step 1: search_notes({'query': 'comet 1'})
  step 2: search_notes({'query': 'comet 2'})
  step 3: search_notes({'query': 'comet 3'})
  step 1: search_notes({'query': 'Pluto'})
  step 2: search_notes({'query': 'Pluto'})
  step 3: search_notes({'query': 'Pluto'})
  step 1: search_notes({'query': 'comet 1'})
  step 2: search_notes({'query': 'comet 2'})


Every run ended with a reason, no answer, and a history the caller can still show or save.

In [14]:
for guard, run in runs.items():
    print(f"{guard:10} -> stop={run['stop']:13} steps={run['steps']} answer={run['answer']} "
          f"messages={len(run['messages'])}")

step limit -> stop=step_limit    steps=4 answer=None messages=9
timeout    -> stop=timeout       steps=3 answer=None messages=7
repeats    -> stop=repeated_call steps=3 answer=None messages=7
cancel     -> stop=cancelled     steps=2 answer=None messages=5


And with a model that behaves, the guards stay out of the way: lesson 26's script still answers.

In [15]:
def last_result(messages): return json.loads(messages[-1]["content"])
MARS_DAY_SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "day"}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda messages: {"text": f"A day on Mars lasts about {last_result(messages)} minutes."},
]
run = run_agent(QUESTION, ScriptedModel(MARS_DAY_SCRIPT), TOOLS, SCHEMAS)
print(run["stop"], "after", run["steps"], "steps:", run["answer"])

  step 1: search_notes({'query': 'day'})
  step 2: calculator({'expression': '24*60+39'})
end_of_turn after 3 steps: A day on Mars lasts about 1479 minutes.


### 8. The same guards in real time
Now a real clock and a real second thread. Each model call sleeps 0.1 s. A timer thread presses Stop
after 0.35 s. The exact step count can vary by one from machine to machine, so we print only the
reason and check the time (`verbose=False` hides the step lines).

In [16]:
class SleepyModel(ScriptedModel):
    def __call__(self, messages, tools):
        time.sleep(0.1)
        return super().__call__(messages, tools)

stop_button = threading.Event()
threading.Timer(0.35, stop_button.set).start()                # the user clicks Stop at 0.35 s
run = run_agent(QUESTION, SleepyModel(WANDERING), TOOLS, SCHEMAS, max_steps=50, cancel=stop_button,
                verbose=False)
print(run["stop"], "| under 1 s:", run["elapsed"] < 1)
run = run_agent(QUESTION, SleepyModel(WANDERING), TOOLS, SCHEMAS, max_steps=50, timeout_s=0.25,
                verbose=False)
print(run["stop"], "| under 1 s:", run["elapsed"] < 1)

cancelled | under 1 s: True


timeout | under 1 s: True


### 9. Gemini, with a timeout on each call
The deadline cannot interrupt a call that hangs. So each Gemini call also gets its own **per-call
timeout**: `types.HttpOptions(timeout=...)`, in milliseconds, inside the request config. If the server
does not answer in time the SDK raises instead of waiting forever. Lesson 26's `to_contents`, with
lesson 28's error results, and `GeminiModel` with that one addition:

In [17]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                        # unchanged: keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper counts its calls too, so the cancel helper from step 7 works with it.

In [18]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Do every calculation with the calculator tool, never in your head. Answer in one sentence.")

class GeminiModel:
    def __init__(self, call_timeout_s: float = 30):
        self.http, self.calls = types.HttpOptions(timeout=int(call_timeout_s * 1000)), 0

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        self.calls += 1
        decls = [types.FunctionDeclaration(name=t["name"], description=t["description"],
                                           parameters_json_schema=t["parameters"]) for t in tools]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=[types.Tool(function_declarations=decls)],
                                          automatic_function_calling=NO_AFC, http_options=self.http)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

**What a per-call timeout looks like.** We cannot make Gemini hang on purpose, but we checked the SDK
against a local socket that never answers: with `HttpOptions(timeout=300)` it raised `httpx.ReadTimeout`
after about 0.3 s. This stub behaves the same way: it waits out a 0.2 s timeout, then raises.

In [19]:
import httpx

class HangingModel:
    def __init__(self, call_timeout_s: float):
        self.call_timeout_s, self.calls = call_timeout_s, 0
    def __call__(self, messages, tools):
        self.calls += 1
        time.sleep(self.call_timeout_s)                   # the server never answers...
        raise httpx.ReadTimeout("timed out")              # ...so the SDK gives up, as it would

start = time.monotonic()
try:
    run = run_agent(QUESTION, HangingModel(0.2), TOOLS, SCHEMAS)
except httpx.TimeoutException as e:
    print(f"caller caught {type(e).__name__} after {time.monotonic() - start:.1f} s, not forever")

caller caught ReadTimeout after 0.2 s, not forever


The wait is bounded, but `run_agent` does not catch the error, so this run's history is lost. Retrying
the call and keeping the work is lesson 15's failure drill.

### 10. Gemini under the guards
First a normal run: all four guards on, none should fire.

In [20]:
run = run_agent(QUESTION, GeminiModel(call_timeout_s=30), TOOLS, SCHEMAS, max_steps=6, timeout_s=60)
print(run["stop"], "after", run["steps"], "steps:", run["answer"])

  step 1: search_notes({'query': 'Mars'})
  step 2: calculator({'expression': '24 * 60 + 39'})
end_of_turn after 3 steps: A day on Mars is 1,479 minutes long.


Now the user presses Stop while Gemini's first call is running. The loop finishes that step (it runs
the tool Gemini asked for) and stops before the second call. Every tool call in the history has its
result, so the run could be shown, saved, or resumed later.

In [21]:
stop_button, gemini = threading.Event(), GeminiModel()
run = run_agent(QUESTION, press_stop_on_call(gemini, 1, stop_button), TOOLS, SCHEMAS, cancel=stop_button)
print(run["stop"], "| model calls:", gemini.calls, "| answer:", run["answer"])
asked = [c["id"] for m in run["messages"] for c in m.get("tool_calls", [])]
answered = [m["call_id"] for m in run["messages"] if m["role"] == "tool"]
print("every call answered:", asked == answered)

  step 1: search_notes({'query': 'Mars'})
cancelled | model calls: 1 | answer: None
every call answered: True


## At work
- **Runaway runs cost real money**, so production agents cap steps, time *and* spend per run, and cut
  off a run that keeps paying for the same call. → lesson 54
- **Vendors put guards in their own loops too.** Gemini has a `TOO_MANY_TOOL_CALLS` finish reason, and
  Anthropic returns `pause_turn` when a server-run tool loop hits its iteration limit; you send the
  reply back to continue. → lesson 44
- **The google-genai client does not retry by default and a hung call blocks the loop**, so teams set
  a per-call timeout and test what happens when the provider stalls. → lesson 15
- **Every run's stop reason is logged.** A rise in `timeout` or `repeated_call` on a dashboard is often
  the first sign that a prompt or tool change broke something. → lesson 49

## What just happened
- A **wall-clock timeout** (`Deadline`, on `time.monotonic`) stops a run that is too slow, checked
  between steps; a **per-call timeout** (`HttpOptions(timeout=...)`) makes one hung call raise
  `httpx.ReadTimeout` (shown with a stub); handling that error well is lesson 15.
- **Loop detection** (`RepeatWatch`) counts identical calls by a sorted-JSON key and stops the stuck
  model at its third repeat, long before the step limit.
- **Cancellation** is a `threading.Event` the loop checks at the start of each step, so a Stop button
  on another thread can end the run.
- Every way out returns the same dict: `stop` names the reason, and `messages` keeps the work so far,
  with every tool call answered.
- A fake clock made the timeout testable in no time; Gemini ran normally with the guards on.

## Common mistakes
**1. Putting the calls themselves in a set.** It looks like the natural way to spot a repeat, but a
dict is not hashable.

In [22]:
seen_calls = set()
for call in STUCK[0]["tool_calls"] * 3:
    if call in seen_calls:
        print("repeat!")
    seen_calls.add(call)

TypeError: unhashable type: 'dict'

Fix: count a string key built with `json.dumps(..., sort_keys=True)`, as `call_key` does.

In [23]:
print(Counter(call_key(c) for c in STUCK[0]["tool_calls"] * 3))

Counter({'search_notes{"query": "Pluto"}': 3})


**2. Raising an exception to stop.** The caller gets a traceback and loses the history: the searches
already done, the tool results, everything it could have shown the user.

In [24]:
def run_agent_raising(question, model, tools, schemas, timeout_s, clock):
    messages, deadline = [{"role": "user", "text": question}], Deadline(timeout_s, clock)
    while True:
        if deadline.expired():
            raise TimeoutError(f"agent ran longer than {timeout_s} s")
        reply = model(messages, schemas)
        messages.append(reply)
        for call in reply.get("tool_calls", []):
            result = run_tool(call, tools)
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})

clock = FakeClock()
result = run_agent_raising(QUESTION, SlowModel(ScriptedModel(WANDERING), clock, 7), TOOLS, SCHEMAS, 20, clock)

TimeoutError: agent ran longer than 20 s

Fix: return a result with a `stop` reason, as `run_agent` does. The caller decides what to do, and
the work is still there.

In [25]:
clock = FakeClock()
run = run_agent(QUESTION, SlowModel(ScriptedModel(WANDERING), clock, 7), TOOLS, SCHEMAS, timeout_s=20, clock=clock)
print(run["stop"], "| kept", len(run["messages"]), "messages | last tool call:",
      run["messages"][-2]["tool_calls"][0]["args"])

  step 1: search_notes({'query': 'comet 1'})
  step 2: search_notes({'query': 'comet 2'})
  step 3: search_notes({'query': 'comet 3'})
timeout | kept 7 messages | last tool call: {'query': 'comet 3'}


## Try it
Add a fifth guard: stop with `stop="too_many_errors"` once the tools have returned more than
`max_errors` errors in one run (default 3). Test it with a fake model that keeps calling
`calculator` with `"2^3"` (the `^` is not allowed, so every call is an error). Which guard fires first
if you also leave `max_repeats=2`, and why?

In [26]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [27]:
llm_replay.stop()

## Key terms
- **stopping condition** — a rule that ends the agent loop: end of turn, step limit, timeout, repeated call or cancel; the run reports which one as `stop`.
- **wall-clock timeout** — a limit on the real time a whole run may take, checked between steps against `time.monotonic()`.
- **per-call timeout** — a limit on one model request (`types.HttpOptions(timeout=ms)` in google-genai), so a hung call raises instead of blocking the loop.
- **loop detection** — stopping a run when the model makes the same tool call (same name, same arguments) more than an allowed number of times.
- **cancellation** — ending a run on request from outside the loop, e.g. a Stop button setting a `threading.Event` the loop checks each step.
- **monotonic clock** — a clock that only moves forward (`time.monotonic()`), safe for measuring durations even when the system time is changed.